# GIAI ĐOẠN 1: TIỀN XỬ LÝ VÀ LÀM SẠCH DỮ LIỆU (DATA CLEANING)

**Đề tài:** Phân tích rủi ro rời bỏ dịch vụ thẻ tín dụng (Customer Churn Analysis)
**Mục tiêu của Notebook:**
1. Khám phá hiện trạng, kiểm định chất lượng tệp dữ liệu thô.
2. Thực thi các quyết định làm sạch trên bản sao độc lập nhằm bảo toàn 100% dữ liệu gốc.
3. Rà soát logic miền giá trị, định lượng ngoại lai và chuẩn hóa các biến đặc trưng.
4. Lập bảng đối chiếu chứng minh chất lượng dữ liệu được cải thiện trước khi đưa vào phân tích chuyên sâu.

In [1]:
import pandas as pd
import numpy as np 

In [2]:
df_raw=pd.read_csv('BankChurners.csv')
df_clean=df_raw.copy()

## 1. Khám phá Tổng quan Dữ liệu (Data Audit)
Bước đầu tiên là kiểm tra kích thước tập dữ liệu, rà soát các dòng trùng lặp và phát hiện các giá trị khuyết thiếu. Đặc biệt, ta cần quét các cột dạng chuỗi (Object) để tìm các giá trị khuyết bị ẩn dưới dạng nhãn văn bản (ví dụ: 'Unknown').

In [3]:
df_raw.head()
print(f"Dữ liệu gồm: {df_raw.shape[0]} dòng, {df_raw.shape[1]} cột")
print(f"Số dữ liệu trùng lặp: {df_raw.duplicated().sum()} dòng")
print(f"Số ô NaN (null) theo kỹ thuật: {df_raw.isna().sum().sum()}")

Dữ liệu gồm: 10127 dòng, 23 cột
Số dữ liệu trùng lặp: 0 dòng
Số ô NaN (null) theo kỹ thuật: 0


In [4]:
#Đếm ô unknown
name_col=df_raw.columns
for i in name_col:
    dem_unk=(df_raw[i]=='Unknown').sum()
    if dem_unk>0:
        print(f"Cột {i} có: {dem_unk} ô Unknown ({dem_unk/len(df_raw):.2%})")
#Đếm số khách hàng chứa ít nhất 1 thông tin Unkown
so_data_unk=(df_raw[name_col]=='Unknown').any(axis=1).sum()
print(f"Số khách thiếu thông tin: {so_data_unk}/{len(df_raw)} ({so_data_unk/len(df_raw):.2%})")



Cột Education_Level có: 1519 ô Unknown (15.00%)
Cột Marital_Status có: 749 ô Unknown (7.40%)
Cột Income_Category có: 1112 ô Unknown (10.98%)
Số khách thiếu thông tin: 3046/10127 (30.08%)


## 2. Các Quyết định Xử lý Dữ liệu & Biện giải Nghiệp vụ
Dựa trên kết quả khám phá, các thao tác làm sạch sau được thực thi:

1. **Xử lý cột CLIENTNUM:** Chuyển từ biến đặc trưng (Feature) thành Khóa dòng (Index). Điều này giúp thuật toán học máy sau này không bị "học vẹt" (overfitting) theo mã định danh, nhưng vẫn giữ lại được ID để truy xuất hồ sơ trong hệ thống cơ sở dữ liệu.
2. **Triệt tiêu nguy cơ rò rỉ dữ liệu (Data Leakage):** Loại bỏ hoàn toàn 2 cột `Naive_Bayes_Classifier_...` ở cuối bảng. Đây là xác suất dự đoán từ một mô hình bên ngoài, việc giữ lại sẽ làm hỏng tính khách quan của bài toán.
3. **Chuẩn hóa biến mục tiêu (`Attrition_Flag`):** Mã hóa nhị phân (`Existing Customer` -> 0, `Attrited Customer` -> 1) và ép kiểu `int8` để tối ưu bộ nhớ RAM, sẵn sàng cho tính toán ma trận tương quan.
4. **Chiến lược giữ nguyên nhãn 'Unknown':** Với hơn 30% mẫu quan sát chứa nhãn 'Unknown', việc xóa dòng sẽ làm sụp đổ sức mạnh thống kê của dữ liệu. Ta quyết định giữ nguyên 'Unknown' thành một nhóm danh mục độc lập để mô hình có thể khai thác "hành vi từ chối cung cấp thông tin" của khách hàng.

In [5]:
#1: Chuyển CLIENTNUM thành index
df_clean=df_clean.set_index('CLIENTNUM')
#2: Xóa 2 cột gây rò rỉ dữ liệu Naive_Bayes
leak_col=[c for c in df_clean.columns if c.startswith('Naive_Bayes')]
df_clean=df_clean.drop(columns=leak_col)
#3: Chuyển biến mục tiêu Attrition_Flag thành 0/1
df_clean['Attrition_Flag']=df_clean['Attrition_Flag'].map({
    'Existing Customer':0,
    'Attrited Customer':1
}).astype('int8')

## 3. Kiểm định Logic miền giá trị và Đánh giá Ngoại lai (Outliers)
Hệ thống sẽ tự động quét toàn bộ các biến định lượng để:
- Đảm bảo không có vi phạm logic vật lý (ví dụ: tuổi âm, số tiền âm, lỗi chia cho 0).
- Sử dụng quy tắc hàng rào Tứ phân vị ($1.5 \times \text{IQR}$) và hệ số Skewness để định lượng các điểm bất thường.

In [6]:
cols=[c for c in (df_clean.select_dtypes(include=['number']).columns)
         if c != 'Attrition_Flag']
tao_bang=[]
for col in cols:
    s=df_clean[col]
    #1 Kiểm tra logic cận biên
    check_neg=int((s<0).sum())
    check_infi=int((~np.isfinite(s)).sum())
    #2 Kiểm tra các giá trị ngoại lai
    q1=s.quantile(0.25)
    q3=s.quantile(0.75)
    iqr=q3-q1
    lower_bound=q1-1.5*iqr
    upper_bound=q3+1.5*iqr
    num_outlier = int(((s < lower_bound) | (s > upper_bound)).sum())
    pr_outlier = f"{num_outlier / len(df_clean):.2%}"

    tao_bang.append({
    'Thuộc tính':col,
    'Min':round(s.min(),2),
    'Max':round(s.max(),2),
    'Số giá trị âm':check_neg,
    'Số giá trị vô cùng':check_infi,
    'Độ lệch (Skew)':round(s.skew(),2),
    'Số giá trị ngoại lai':num_outlier,
    'Tỉ lệ ngoại lai':pr_outlier
    })
# Xuất bảng tổng hợp kết quả
df_report = pd.DataFrame(tao_bang)
print("--- BẢNG KIỂM ĐỊNH TÍNH HỢP LỆ VÀ NGOẠI LAI ---")
display(df_report.sort_values(by="Số giá trị ngoại lai", ascending=False).reset_index(drop=True))

--- BẢNG KIỂM ĐỊNH TÍNH HỢP LỆ VÀ NGOẠI LAI ---


,Thuộc tính,Min,Max,Số giá trị âm,Số giá trị vô cùng,Độ lệch (Skew),Số giá trị ngoại lai,Tỉ lệ ngoại lai
0,Credit_Limit,1438.3,34516.00,0,0,1.67,984,9.72%
1,Avg_Open_To_Buy,3.0,34516.00,0,0,1.66,963,9.51%
2,Total_Trans_Amt,510.0,18484.00,0,0,2.04,896,8.85%
3,Contacts_Count_12_mon,0.0,6.00,0,0,0.01,629,6.21%
4,Total_Amt_Chng_Q4_Q1,0.0,3.40,0,0,1.73,396,3.91%
5,Total_Ct_Chng_Q4_Q1,0.0,3.71,0,0,2.06,394,3.89%
6,Months_on_book,13.0,56.00,0,0,-0.11,386,3.81%
7,Months_Inactive_12_mon,0.0,6.00,0,0,0.63,331,3.27%
8,Customer_Age,26.0,73.00,0,0,-0.03,2,0.02%
9,Total_Trans_Ct,10.0,139.00,0,0,0.15,2,0.02%


 **Kết luận kiểm định Ngoại lai & Logic:**
 1. **Về tính hợp lệ logic:** 100% các quan sát đều là số thực hữu hạn. Không xuất hiện giá trị âm ở bất kỳ biến tài chính hay nhân khẩu học nào (tuổi, hạn mức, số dư đều hợp lý).
 2. **Về phân phối và ngoại lai:** 
    - Các biến tần suất (`Total_Trans_Ct`, `Customer_Age`) có phân phối chuẩn, tỷ lệ ngoại lai gần như bằng 0.
    - Các biến tài chính (`Credit_Limit`, `Total_Trans_Amt`) có độ lệch phải mạnh (Skewness > 1.6) với tỷ lệ vượt hàng rào IQR từ 8.8% đến 9.7%.
 3. **Quyết định xử lý nghiệp vụ:** Bảo toàn nguyên vẹn 100% các điểm "ngoại lai" này. Đây không phải là lỗi nhập liệu mà phản ánh hành vi thực tế của nhóm khách hàng phân khúc giá trị cao (VIP) với hạn mức chạm trần hoặc chi tiêu đột biến. Việc xử lý độ lệch phân phối sẽ được đảm nhiệm bằng các kỹ thuật chuẩn hóa (Scaler) hoặc các mô hình học máy dạng cây ở giai đoạn xây dựng mô hình.

## 4. Bằng chứng Định lượng: Đối chiếu Trước và Sau (Before vs. After)
So sánh bộ dữ liệu gốc (`df_raw`) và bộ dữ liệu đã làm sạch (`df_clean`) để chứng minh tính hiệu quả của quy trình tiền xử lý.

In [7]:
comparison_df = pd.DataFrame({
    "Tiêu chí": [
        "Số dòng (Rows)", 
        "Số cột đặc trưng (Features)", 
        "Cột định danh (ID)",
        "Số dòng trùng lặp", 
        "Số cột rò rỉ đã loại bỏ (Naive Bayes)", 
        "Kiểu dữ liệu biến mục tiêu", 
        "Bộ nhớ RAM (MB)"
    ],
    "df_raw (Trước)": [
        len(df_raw), 
        df_raw.shape[1], 
        "Nằm trong bảng dữ liệu (Feature)",
        df_raw.duplicated().sum(), 
        0, 
        str(df_raw["Attrition_Flag"].dtype), 
        round(df_raw.memory_usage(deep=True).sum() / 1024**2, 2)
    ],
    "df_clean (Sau)": [
        len(df_clean), 
        df_clean.shape[1], 
        "Đã chuyển thành Index (Khóa dòng)",
        df_clean.index.duplicated().sum(), 
        len(leak_col), 
        str(df_clean["Attrition_Flag"].dtype), 
        round(df_clean.memory_usage(deep=True).sum() / 1024**2, 2)
    ]
})

print("\n--- BẢNG ĐỐI CHIẾU BEFORE vs AFTER ---")
print(comparison_df.to_string(index=False))

# Xuất dữ liệu sạch ra tệp tin (Lưu ý: index=True để giữ lại cột CLIENTNUM làm khóa)
df_clean.to_csv("BankChurners_clean.csv", index=True)
print("\n-> Đã xuất tệp 'BankChurners_clean.csv' thành công.")



--- BẢNG ĐỐI CHIẾU BEFORE vs AFTER ---
                             Tiêu chí                   df_raw (Trước)                    df_clean (Sau)
                       Số dòng (Rows)                            10127                             10127
          Số cột đặc trưng (Features)                               23                                20
                   Cột định danh (ID) Nằm trong bảng dữ liệu (Feature) Đã chuyển thành Index (Khóa dòng)
                    Số dòng trùng lặp                                0                                 0
Số cột rò rỉ đã loại bỏ (Naive Bayes)                                0                                 2
           Kiểu dữ liệu biến mục tiêu                              str                              int8
                      Bộ nhớ RAM (MB)                             4.63                               4.1

-> Đã xuất tệp 'BankChurners_clean.csv' thành công.
